<a href="https://colab.research.google.com/github/MuhammadAqsandy/scikit-learn-cookbook/blob/main/chapter_09_text_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 9: Text Processing and Multiclass Classification
## 📌 Summary
Text processing mengubah raw text menjadi representasi numerik yang bisa digunakan model ML. Teknik utama: **CountVectorizer**, **TF-IDF**, dan **text pipelines**.

## 🧠 Theoretical Deep-Dive

### 9.1 Bag of Words (BoW)
Representasikan dokumen sebagai vektor frekuensi kata. Mengabaikan urutan kata.
`CountVectorizer`: menghasilkan raw count matrix.

### 9.2 TF-IDF
TF-IDF = TF × IDF
- **TF** (Term Frequency): frekuensi kata dalam dokumen
- **IDF** (Inverse Document Frequency): log(N/df) → kata yang jarang = lebih informatif

TF-IDF menurunkan bobot kata umum (the, is, a) dan menaikkan kata yang distinctive.

### 9.3 N-grams
Extend BoW dengan urutan kata: unigram (1 kata), bigram (2 kata), trigram (3 kata). Menangkap konteks lokal.

### 9.4 Multiclass Strategies
- **OneVsRest**: k classifiers, satu per kelas
- **OneVsOne**: k(k-1)/2 classifiers, semua pasangan kelas
- **Multinomial NB**: native multiclass, dirancang untuk text

### 9.5 Naive Bayes untuk Text
P(class|text) ∝ P(class) × Π P(word|class)
Asumsi: kata independent given class (Naive). Tapi sangat efektif untuk text classification.

## 💻 Code Reproduction

In [1]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "machine learning is fun",
    "learning python for machine learning",
    "python is great for data science",
    "deep learning and machine learning"
]

cv = CountVectorizer()
X = cv.fit_transform(corpus)

print("Vocabulary:", cv.vocabulary_)
print("Feature names:", cv.get_feature_names_out())
print("Document-term matrix (dense):")
print(X.toarray())
print("Shape:", X.shape)

Vocabulary: {'machine': 8, 'learning': 7, 'is': 6, 'fun': 4, 'python': 9, 'for': 3, 'great': 5, 'data': 1, 'science': 10, 'deep': 2, 'and': 0}
Feature names: ['and' 'data' 'deep' 'for' 'fun' 'great' 'is' 'learning' 'machine'
 'python' 'science']
Document-term matrix (dense):
[[0 0 0 0 1 0 1 1 1 0 0]
 [0 0 0 1 0 0 0 2 1 1 0]
 [0 1 0 1 0 1 1 0 0 1 1]
 [1 0 1 0 0 0 0 2 1 0 0]]
Shape: (4, 11)


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

corpus = [
    "machine learning is fun",
    "learning python for machine learning",
    "python is great for data science",
    "deep learning and machine learning"
]

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)

print("Feature names:", tfidf.get_feature_names_out())
print("\nTF-IDF matrix:")
print(X_tfidf.toarray().round(3))
print("\nHighest TF-IDF words per doc:")
for i, doc in enumerate(corpus):
    scores = X_tfidf[i].toarray()[0]
    top_idx = np.argsort(scores)[::-1][:3]
    top_words = [(tfidf.get_feature_names_out()[j], scores[j].round(3)) for j in top_idx if scores[j] > 0]
    print(f"Doc {i}: {top_words}")

Feature names: ['and' 'data' 'deep' 'for' 'fun' 'great' 'is' 'learning' 'machine'
 'python' 'science']

TF-IDF matrix:
[[0.    0.    0.    0.    0.641 0.    0.505 0.409 0.409 0.    0.   ]
 [0.    0.    0.    0.435 0.    0.    0.    0.705 0.352 0.435 0.   ]
 [0.    0.453 0.    0.357 0.    0.453 0.357 0.    0.    0.357 0.453]
 [0.498 0.    0.498 0.    0.    0.    0.    0.635 0.318 0.    0.   ]]

Highest TF-IDF words per doc:
Doc 0: [('fun', np.float64(0.641)), ('is', np.float64(0.505)), ('machine', np.float64(0.409))]
Doc 1: [('learning', np.float64(0.705)), ('for', np.float64(0.435)), ('python', np.float64(0.435))]
Doc 2: [('science', np.float64(0.453)), ('data', np.float64(0.453)), ('great', np.float64(0.453))]
Doc 3: [('learning', np.float64(0.635)), ('deep', np.float64(0.498)), ('and', np.float64(0.498))]


In [3]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

categories = ["sci.space", "rec.sport.hockey", "comp.graphics", "talk.politics.misc"]
train = fetch_20newsgroups(subset="train", categories=categories, remove=("headers", "footers", "quotes"))
test = fetch_20newsgroups(subset="test", categories=categories, remove=("headers", "footers", "quotes"))

print("Train samples:", len(train.data))
print("Test samples:", len(test.data))
print("Categories:", train.target_names)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words="english")),
    ("clf", MultinomialNB(alpha=0.1))
])

pipe.fit(train.data, train.target)
pred = pipe.predict(test.data)
print("\nAccuracy:", (pred == test.target).mean().round(4))
print("\nClassification Report:")
print(classification_report(test.target, pred, target_names=categories))

Train samples: 2242
Test samples: 1492
Categories: ['comp.graphics', 'rec.sport.hockey', 'sci.space', 'talk.politics.misc']

Accuracy: 0.8854

Classification Report:
                    precision    recall  f1-score   support

         sci.space       0.93      0.91      0.92       389
  rec.sport.hockey       0.88      0.96      0.92       399
     comp.graphics       0.87      0.82      0.84       394
talk.politics.misc       0.86      0.84      0.85       310

          accuracy                           0.89      1492
         macro avg       0.88      0.88      0.88      1492
      weighted avg       0.89      0.89      0.88      1492



In [4]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

categories = ["sci.space", "rec.sport.hockey", "comp.graphics", "talk.politics.misc"]
train = fetch_20newsgroups(subset="train", categories=categories, remove=("headers","footers","quotes"))
test = fetch_20newsgroups(subset="test", categories=categories, remove=("headers","footers","quotes"))

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words="english")

classifiers = [
    ("Naive Bayes", MultinomialNB(alpha=0.1)),
    ("Logistic Regression", LogisticRegression(max_iter=1000, C=10)),
    ("Linear SVC", LinearSVC(C=1.0, max_iter=2000))
]

for name, clf in classifiers:
    pipe = Pipeline([("tfidf", tfidf), ("clf", clf)])
    pipe.fit(train.data, train.target)
    acc = pipe.score(test.data, test.target)
    print(f"{name:25}: accuracy={acc:.4f}")

Naive Bayes              : accuracy=0.8854
Logistic Regression      : accuracy=0.8787
Linear SVC               : accuracy=0.8747


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

corpus = [
    "machine learning is exciting and fun",
    "python is great for machine learning algorithms",
    "deep learning neural networks are powerful"
]

# N-gram comparison
for ngram_range in [(1,1), (1,2), (2,2)]:
    tfidf = TfidfVectorizer(ngram_range=ngram_range, max_features=20)
    X = tfidf.fit_transform(corpus)
    print(f"n-gram={ngram_range}: {len(tfidf.get_feature_names_out())} features")
    print("Features:", tfidf.get_feature_names_out()[:10], "...\n")

n-gram=(1, 1): 15 features
Features: ['algorithms' 'and' 'are' 'deep' 'exciting' 'for' 'fun' 'great' 'is'
 'learning'] ...

n-gram=(1, 2): 20 features
Features: ['algorithms' 'and' 'and fun' 'are' 'are powerful' 'deep' 'deep learning'
 'exciting' 'exciting and' 'for'] ...

n-gram=(2, 2): 15 features
Features: ['and fun' 'are powerful' 'deep learning' 'exciting and' 'for machine'
 'great for' 'is exciting' 'is great' 'learning algorithms' 'learning is'] ...



## ✅ Chapter Summary
- **CountVectorizer** → raw counts; **TF-IDF** → penalized by document frequency
- **N-grams** menangkap konteks lokal (bigrams sering meningkatkan performa)
- **Multinomial Naive Bayes** → sangat efisien dan efektif untuk text (baseline kuat)
- **Linear SVC** + TF-IDF → sering best untuk text classification
- Gunakan `stop_words='english'` dan `max_features` untuk efisiensi
- Pipeline text+classifier → mudah deploy dan evaluate